# GEFSv12 week-four temperature reforecasts
Install `.[gefs,notebooks]` in this notebook's Python environment. Downloads are anonymous; no NOAA, IRI, ECMWF, CDS or AWS key is required. Allow `noaa-gefs-retrospective.s3.amazonaws.com` if your network restricts destinations.

Start with this single-week pilot: its January 5, 2000 initialization and extended-range control/member-10 files were verified against the public archive. `DRY_RUN=True` lists which target weeks match archive metadata; set it to False to download. After the pilot completes, expand START/END (for example 2000-01-01 through 2020-02-01). Dates filter target-week starts; END is exclusive. Initializations are limited to 2000–2019, so early January 2000 and later targets may be unmatched.

TARGETS contains ERA5 weekly CSVs with `timestamp`, `lat`, `lon`; changing it changes the processing region automatically. Choose separate OUTPUT/STATE folders for separate studies. The planner checks listed member directories and the control index; all eleven member fields are verified during retrieval. It never silently shifts a target week.

LEAD_DAYS=26 selects [initialization +26 days, +33 days), exactly 28 six-hourly instantaneous temperatures per member. The output mean is **not** a reconstructed hourly mean. For strict verification, aggregate hourly ERA5 at the same valid times stored in `samples.parquet`. Interpolation maps the native forecast (0.5° in the verified pilot) to your ERA5 coordinates; it does not create additional forecast resolution.

The client uses `.grib2.idx` offsets and verified HTTP 206 ranges to download only the requested temperature messages. If an index cannot describe all requested steps exactly once, it downloads that member's full temperature file and verifies the actual GRIB fields instead. This uses more bandwidth for that member; genuinely missing/duplicate samples still fail validation and never produce a partial weekly mean. Completed weeks resume using checksum-verified manifests. KEEP_RAW=False removes global GRIB slices only after both regional sample and weekly files are committed; source URLs, ETags, ranges and checksums remain in each manifest. Set KEEP_RAW=True to retain global slices too (much larger disk use). Interrupted weeks retain their raw cache for retry.

A download returns `result["files"]` (weekly/sample paths), `result["coverage"]`, counts and transferred bytes. The current file list and coverage are saved under STATE; old output versions can remain on disk, so use that list rather than globbing all historical plans. `samples.parquet` keeps member, valid_time, forecast_hour, lat/lon and temperature_k; `weekly.parquet` keeps control and perturbed_1 through perturbed_10 in K. Neither the notebook nor provider alters your ERA5 actuals.

`ON_INCOMPLETE="raise"` stops on incomplete source data. Set it to `"record"` to record a whole week as `unavailable` in coverage and continue with later weeks, without writing any partial eleven-member row. This applies only after actual GRIB metadata has been validated and requested hours are missing; authentication/network failures, wrong members/initializations, duplicate hours, and unexpected formats still raise errors. Unavailable weeks retain raw cache files for inspection. The result reports an `unavailable` count separately from cached/completed weeks.


In [ ]:
from datetime import date
from comm_research.markets.weather import gefs

START = date(2000, 1, 31)
END = date(2000, 2, 7)  # Exclusive; one verified pilot week first.
LEAD_DAYS = 26
WORKERS = 4
DRY_RUN = True
ON_INCOMPLETE = "raise"  # Or "record": mark whole incomplete weeks unavailable and continue.
REFRESH = False
REFRESH_CATALOGUE = False
KEEP_RAW = False
MIN_FREE_GB = 2.0

TARGETS = gefs.TARGETS
OUTPUT = gefs.OUTPUT
RAW = gefs.RAW
STATE = gefs.STATE


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

result = gefs.run(targets=TARGETS, output=OUTPUT, raw=RAW, state=STATE,
                  start=START, end=END, lead_days=LEAD_DAYS, workers=WORKERS,
                  dry_run=DRY_RUN, refresh=REFRESH, refresh_catalogue=REFRESH_CATALOGUE,
                  keep_raw=KEEP_RAW, min_free_gb=MIN_FREE_GB, on_incomplete=ON_INCOMPLETE)
result if DRY_RUN else result["coverage"]
